# Web Scraping para Baixar Fontes de Dados

## Configuração do Ambiente

In [0]:
%sh

pip install playwright

In [ ]:
!playwright install
!playwright install-deps

In [0]:
%restart_python

In [0]:
from playwright.async_api import async_playwright
import asyncio
import unicodedata
import re
import requests

## Script de Raspagem de Dados (Web Scraping)

In [0]:
async def fetch_file_names_and_download():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()
        await page.goto("https://dados.gov.br/dados/conjuntos-dados/grandes-nmeros-do-imposto-de-renda-da-pessoa-fsica")
        await asyncio.sleep(10)
        rows = await page.query_selector_all("div.row.flex.mb-5")
        file_names = []
        for row in rows:
            span = await row.query_selector('span.resource-icon-left')
            if span:
                span_text = await span.inner_text()
                if span_text.strip() == "CSV":
                    h4 = await row.query_selector("h4")
                    if h4:
                        text = await h4.inner_text()
                        if "Faixa de Rendimento" in text:
                            text = f"{text}-em-salarios-minimos"    
                        else:
                            text = f"{text}"
                        file_names.append(text)
        await browser.close()
    return file_names

In [0]:
file_names = await fetch_file_names_and_download()

## Formatação da URL de Download das Fontes de Dados

In [ ]:
import os
import re
import unicodedata

directory = "/Volumes/fiap_1ctor/data_sources/data_sources"

try:
    files = dbutils.fs.ls(directory)

    file_names = [
        f.name.replace(".csv", "")
        for f in files
        if f.name.endswith(".csv")
    ]

    print("Arquivos encontrados no volume:")
    print(file_names)

except Exception as e:

    print("Diretório não encontrado:", directory)

    file_names = [
        'bens-e-direitos',
        'capital-de-estado-de-residencia-do-declarante',
        'dividas-e-onus',
        'estado-de-residencia-do-declarante',
        'faixa-de-base-de-calculo-anual',
        'faixa-de-base-de-calculo-em-salarios-minimos-e-genero',
        'faixa-de-doacoes-e-herancas',
        'faixa-de-rendimentos-totais',
        'faixa-de-rendimentos-tributaveis-tributacao-exclusiva-em-salarios-minimos',
        'faixa-de-rendimento-tributavel-bruto-em-salarios-minimos',
        'faixa-etaria-do-declarante-e-genero',
        'genero-e-tipo-de-declaracao',
        'municipio-de-residencia-do-declarante-e-tipo-de-formulario',
        'natureza-de-ocupacao',
        'ocupacao-principal-do-declarante',
        'pagamentos-e-doacoes',
        'recebedores-de-lucros-e-dividendos-rend-socio-e-titular-microempresa-por-faixa-de-rendimento-total-em-salarios-minimos',
        'recebedores-de-lucros-e-dividendos-rend-socio-e-titular-microempresa-por-ocupacao-principal',
        'rendimentos-isentos-e-nao-tributaveis',
        'rendimentos-sujeitos-a-tributacao-exclusiva-definitiva',
        'rendimentos-tributaveis-por-faixa-de-salarios-minimos',
        'situacao-fiscal',
        'tipo-de-formulario'
    ]

In [0]:
def normalize_filename(name):
    name = name.lower()
    name = name.replace(' ', '-')
    name = name.replace('ç', 'c')
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = re.sub(r'[^a-zA-Z0-9\-:/]', '-', name)
    if (name == "rendimentos-sujeitos-a-tributacao-exclusiva-definitiva"):
        name = "rendimentos-sujeitos-a-tributacao-exclusiva_definitiva"
    if (name == "faixa-de-rendimentos-totais-em-salarios-minimos"):
        name = "faixa-de-rendimentos-totais"
    if (name == "recebedores-de-lucros-e-dividendos-rend-socio-e-titular-microempresa-por-faixa-de-rendimento-total-em-salarios-minimos"):
        name = "recebedores-de-lucros-e-dividendos-rend-socio-e-titular-microempresa-por-faixa-de-rendimento-total"
    if (name == "recebedores-de-lucros-e-dividendos-rend-socio-e-titular-microempresa-por-faixa-de-rendimento-total-em-salarios-minimos.csv"):
        name = "recebedores-de-lucros-e-dividendos-rend-socio-e-titular-microempresa-por-faixa-de-rendimento-total.csv"
    return name

file_names = [n.replace('.csv', '') for n in file_names]
file_names = [normalize_filename(n) for n in file_names]
file_names = [n.replace('--', '') for n in file_names]

## Baixando as Fontes de Dados e Salvando no Unity Catalog

In [ ]:
import unicodedata
import re

def normalize_column(col):

    col = col.lower().strip()

    col = unicodedata.normalize('NFKD', col)\
        .encode('ASCII', 'ignore')\
        .decode('utf-8')

    col = re.sub(r'[^\w]', '_', col)
    col = re.sub(r'_+', '_', col)

    # limitar tamanho da coluna
    col = col[:120]

    return col

In [0]:
import pandas as pd
import requests
import urllib3
import unicodedata
import re
from io import StringIO

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url_base = "https://www.gov.br/receitafederal/dados/"
fallback_path = "/Workspace/Users/rm360803@fiap.com.br/1ctor/dados"

catalog_name = "dados_gov"
schema_name  = "imposto_de_renda_data_sources"

def normalize_column(col):

    col = col.lower()

    col = unicodedata.normalize("NFKD", col)\
        .encode("ASCII", "ignore")\
        .decode("utf-8")

    col = re.sub(r"[^a-zA-Z0-9_]", "_", col)

    col = re.sub("_+", "_", col)

    col = col.strip("_")

    # 🔴 limite unity catalog
    return col[:200]


print("===== INICIANDO PROCESSO DE INGESTÃO =====")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

headers = {"User-Agent": "Mozilla/5.0"}

for entity_name in file_names:

    table_entity_name = entity_name.replace("-", "_")
    table_name = f"{catalog_name}.{schema_name}.{table_entity_name}"

    file_url = f"{url_base}{entity_name}.csv"
    fallback_file = f"{fallback_path}/{entity_name}.csv"

    print("\n--------------------------------------")
    print(f"Processando: {entity_name}")

    try:

        print(f"Download URL: {file_url}")

        response = requests.get(
            file_url,
            headers=headers,
            verify=False,
            timeout=60
        )

        if response.text and len(response.text) > 100:

            print("Download OK")

            csv_data = StringIO(response.text)

            df = pd.read_csv(
                csv_data,
                sep=";",
                engine="python",
                on_bad_lines="skip"
            )

            print(f"Pandas carregado | linhas {len(df)}")

        else:

            raise Exception("Arquivo vazio")

    except Exception as e:

        print("Falha download → fallback")

        df = pd.read_csv(
            fallback_file,
            sep=";",
            engine="python",
            on_bad_lines="skip"
        )

        print(f"Fallback carregado | linhas {len(df)}")


    print("Convertendo Pandas → Spark")

    s_df = spark.createDataFrame(df)


    print("Normalizando colunas")

    for c in s_df.columns:

        new_name = normalize_column(c)

        s_df = s_df.withColumnRenamed(c, new_name)


    print(f"DROP TABLE {table_name}")

    spark.sql(f"DROP TABLE IF EXISTS {table_name}")


    print("Criando tabela Delta")

    s_df.write.mode("overwrite").saveAsTable(table_name)


    print(f"Tabela criada: {table_name}")


print("\n===== PROCESSO FINALIZADO =====")